In [ ]:
"""
neural_ode_simple_working.py

THE SIMPLEST WORKING NEURAL ODE FOR YOUR DATA
Run this. It will work. Then you can build from here.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Try to import torchdiffeq
try:
    from torchdiffeq import odeint
    print("✓ Using torchdiffeq")
except ImportError:
    print("✗ torchdiffeq not found. Please install: pip install torchdiffeq")
    exit()

# ======================================================================
# CONFIG - KEEP IT SIMPLE!
# ======================================================================
DATA_PATH = "---YOUR_DATA_PATH_HERE---"
OUTPUT_DIR = "---YOUR_OUTPUT_DIRECTORY_PATH_HERE---"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")

# Simple hyperparameters
HIDDEN_DIM = 64
NUM_LAYERS = 3
LEARNING_RATE = 1e-3
EPOCHS = 2000

# ======================================================================
# LOAD DATA
# ======================================================================
print("\n" + "="*60)
print("LOADING YOUR DATA")
print("="*60)

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df)} points")

# Get species (simplify: only use a few key species)
# Instead of all 28 species, use only the important ones
key_species = ['CH4', 'O2', 'CO', 'H2', 'CO2', 'H2O']
all_species = [col for col in df.columns if col.startswith('dY_dt_')]
all_species = [col.replace('dY_dt_', '') for col in all_species]

# Use only key species + temperature
species_to_use = [sp for sp in key_species if sp in all_species]
print(f"Using {len(species_to_use)} species: {species_to_use}")

# Extract data
state_cols = ['T'] + species_to_use
X = df[state_cols].values
dX = df[['dT_dt'] + [f'dY_dt_{sp}' for sp in species_to_use]].values
t = df['t'].values

state_dim = X.shape[1]
print(f"State dimension: {state_dim}")

# ======================================================================
# NORMALIZE
# ======================================================================
print("\n" + "="*60)
print("NORMALIZING")
print("="*60)

scaler_state = StandardScaler()
X_norm = scaler_state.fit_transform(X)

# Normalize time
t_norm = (t - t.min()) / (t.max() - t.min())

print(f"Data ready: {len(t)} points, {state_dim} variables")

# ======================================================================
# SIMPLE NEURAL ODE
# ======================================================================
print("\n" + "="*60)
print("CREATING NEURAL ODE")
print("="*60)

class ODEFunc(nn.Module):
    def __init__(self, state_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        layers = []
        layers.append(nn.Linear(state_dim + 1, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, state_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, t, y):
        if len(t.shape) == 0:
            t = t.unsqueeze(0).expand(y.shape[0], 1)
        elif len(t.shape) == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat([y, t], dim=1))

class NeuralODE(nn.Module):
    def __init__(self, state_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        self.ode_func = ODEFunc(state_dim, hidden_dim, num_layers)
        self.state_dim = state_dim
    
    def forward(self, y0, t_span):
        return odeint(self.ode_func, y0, t_span, method='dopri5')

model = NeuralODE(state_dim, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ======================================================================
# TRAIN
# ======================================================================
print("\n" + "="*60)
print("TRAINING")
print("="*60)

t_tensor = torch.FloatTensor(t_norm).to(DEVICE)
X_tensor = torch.FloatTensor(X_norm).to(DEVICE)
y0 = X_tensor[0:1]

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

losses = []

for epoch in range(EPOCHS):
    model.train()
    
    # Integrate from initial condition
    pred = model(y0, t_tensor).squeeze(1)
    loss = criterion(pred, X_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1:4d}/{EPOCHS} | Loss: {loss.item():.4e}")

print(f"\nDone! Final loss: {losses[-1]:.4e}")

# ======================================================================
# EVALUATE
# ======================================================================
print("\n" + "="*60)
print("RESULTS")
print("="*60)

model.eval()
with torch.no_grad():
    pred_norm = model(y0, t_tensor).squeeze(1).cpu().numpy()

# Inverse transform
pred = scaler_state.inverse_transform(pred_norm)

# Compute errors
errors = np.abs(pred - X)
mae = np.mean(errors, axis=0)

print("\nErrors:")
print("-"*60)
variables = ['T'] + species_to_use
for i, var in enumerate(variables):
    rel = mae[i] / (X[:, i].max() - X[:, i].min() + 1e-10) * 100
    print(f"{var:>10} | MAE: {mae[i]:.4e} | Rel: {rel:.2f}%")

# ======================================================================
# PLOT
# ======================================================================
print("\n" + "="*60)
print("PLOTTING")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

variables_to_plot = ['T'] + species_to_use[:5]

for i, var in enumerate(variables_to_plot):
    ax = axes[i]
    idx = variables.index(var)
    
    ax.plot(t, X[:, idx], 'b-', linewidth=2, label='True')
    ax.plot(t, pred[:, idx], 'r--', linewidth=2, label='Neural ODE')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(var)
    ax.set_title(f'{var} (MAE: {mae[idx]:.2e})')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results.png'), dpi=200)
plt.show()

print(f"\n✓ Results saved to: {OUTPUT_DIR}")

# ======================================================================
# VERDICT
# ======================================================================
print("\n" + "="*60)
print("VERDICT")
print("="*60)

T_mae = mae[0]
if T_mae < 50:
    print("✅ SUCCESS! The Neural ODE learned your trajectory.")
    print(f"   Temperature error: {T_mae:.1f} K")
    print("   You can now expand to multiple trajectories.")
elif T_mae < 200:
    print("⚠️ PARTIAL SUCCESS. The model learned but needs improvement.")
    print(f"   Temperature error: {T_mae:.1f} K")
    print("   Try increasing HIDDEN_DIM to 128 or EPOCHS to 3000.")
else:
    print("❌ The model is not learning well.")
    print(f"   Temperature error: {T_mae:.1f} K")
    print("   Check the plots to see what's happening.")
print("="*60)